## Week 5: Additional Models

Team ds55 member: Yingxin Deng

This week tasks: 
1. Try Decision Tree and Random Forest regressors.
2. Compare their test R² against baseline.
3. Document model behavior (strengths/weaknesses).

In [31]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)


RANDOM_STATE = 420

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [32]:
df = pd.read_csv("../Week3/version2/cleaned_housing_data.csv")

print(df.shape)
df.head()

print("\nDataset distribution:")
print(df["Dataset"].value_counts())

print("\nClosePrice summary by dataset:")
display(
    df.groupby("Dataset")["ClosePrice"].describe()
)

(71208, 985)

Dataset distribution:
Dataset
Train    59192
Test     12016
Name: count, dtype: int64

ClosePrice summary by dataset:


,count,mean,std,min,25%,50%,75%,max
Dataset,,,,,,,,
Test,"12,016.0000","1,309,534.2808","1,678,514.8494","11,900.0000","639,000.0000","930,000.0000","1,500,000.0000","97,972,500.0000"
Train,"59,192.0000","1,244,125.8483","1,350,768.8895","123,600.0000","620,000.0000","880,000.0000","1,400,000.0000","60,000,000.0000"


In [ ]:
# ============================================================
# 1. Separate train and test sets
# ============================================================

train_df = df[df["Dataset"] == "Train"].copy()
test_df = df[df["Dataset"] == "Test"].copy()

X_train = train_df.drop(
    columns=["ClosePrice", "Dataset"]
).copy()

X_test = test_df.drop(
    columns=["ClosePrice", "Dataset"]
).copy()

y_train = train_df["ClosePrice"].copy()
y_test = test_df["ClosePrice"].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nFeature columns match:")
print(X_train.columns.equals(X_test.columns))

X_train shape: (59192, 983)
X_test shape: (12016, 983)
y_train shape: (59192,)
y_test shape: (12016,)

Feature columns match:
True


In [ ]:
# ============================================================
# 2. Pre-modeling checks
# ============================================================

# Check non-numeric columns
non_numeric_cols = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("Non-numeric columns:")
print(non_numeric_cols)

# Check missing values
train_missing = X_train.isna().sum()
train_missing = train_missing[train_missing > 0].sort_values(
    ascending=False
)

test_missing = X_test.isna().sum()
test_missing = test_missing[test_missing > 0].sort_values(
    ascending=False
)

print("\nMissing values in X_train:")
print(train_missing)

print("\nMissing values in X_test:")
print(test_missing)

# Check infinity
train_infinite = np.isinf(
    X_train.select_dtypes(include=[np.number])
).sum().sum()

test_infinite = np.isinf(
    X_test.select_dtypes(include=[np.number])
).sum().sum()

print("\nInfinite values in X_train:", train_infinite)
print("Infinite values in X_test:", test_infinite)

Non-numeric columns:
[]

Missing values in X_train:
Series([], dtype: int64)

Missing values in X_test:
Series([], dtype: int64)

Infinite values in X_train: 0
Infinite values in X_test: 0


In [35]:
# Replace infinity with NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Use training medians only
train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Remaining missing in X_train:", X_train.isna().sum().sum())
print("Remaining missing in X_test:", X_test.isna().sum().sum())

Remaining missing in X_train: 0
Remaining missing in X_test: 0


In [36]:
remaining_non_numeric = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

if remaining_non_numeric:
    raise ValueError(
        f"These columns still need encoding or removal: "
        f"{remaining_non_numeric}"
    )

In [ ]:
# ============================================================
# 3. Development-validation split
# ============================================================

X_development, X_validation, y_development, y_validation = (
    train_test_split(
        X_train,
        y_train,
        test_size=0.20,
        random_state=RANDOM_STATE
    )
)

print("Development shape:", X_development.shape)
print("Validation shape:", X_validation.shape)

print("\nDevelopment target summary:")
print(y_development.describe())

print("\nValidation target summary:")
print(y_validation.describe())

Development shape: (47353, 983)
Validation shape: (11839, 983)

Development target summary:
count       47,353.0000
mean     1,243,044.5616
std      1,337,415.4675
min        123,600.0000
25%        620,000.0000
50%        880,000.0000
75%      1,400,000.0000
max     40,000,000.0000
Name: ClosePrice, dtype: float64

Validation target summary:
count       11,839.0000
mean     1,248,450.7211
std      1,402,958.7232
min        125,000.0000
25%        615,000.0000
50%        875,000.0000
75%      1,400,000.0000
max     60,000,000.0000
Name: ClosePrice, dtype: float64


In [ ]:
# ============================================================
# 4. Evaluation functions
# ============================================================

def calculate_metrics(y_true, y_pred):
    """
    Calculate regression evaluation metrics.
    """

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    errors = y_pred - y_true
    absolute_errors = np.abs(errors)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    valid_percentage_mask = y_true != 0

    percentage_errors = (
        absolute_errors[valid_percentage_mask] /
        np.abs(y_true[valid_percentage_mask])
    ) * 100

    mape = np.mean(percentage_errors)
    mdape = np.median(percentage_errors)

    return {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "MdAPE": mdape
    }


def fit_and_evaluate(
    model,
    model_name,
    X_fit,
    y_fit,
    X_eval,
    y_eval
):
    """
    Fit one model and evaluate it on a separate dataset.
    """

    start_time = time.time()

    model.fit(X_fit, y_fit)

    fit_seconds = time.time() - start_time

    train_pred = model.predict(X_fit)
    eval_pred = model.predict(X_eval)

    train_metrics = calculate_metrics(
        y_fit,
        train_pred
    )

    eval_metrics = calculate_metrics(
        y_eval,
        eval_pred
    )

    result = {
        "Model": model_name,
        "Train R2": train_metrics["R2"],
        "Validation R2": eval_metrics["R2"],
        "Validation RMSE": eval_metrics["RMSE"],
        "Validation MAE": eval_metrics["MAE"],
        "Validation MAPE": eval_metrics["MAPE"],
        "Validation MdAPE": eval_metrics["MdAPE"],
        "Fit Seconds": fit_seconds
    }

    return result, model

In [ ]:
# ============================================================
# 5. Candidate models
# ============================================================

candidate_models = [
    (
        "Baseline Mean",
        DummyRegressor(
            strategy="mean"
        )
    ),

    (
        "Week 4 Linear Regression",
        LinearRegression()
    ),

    (
        "Decision Tree depth=10 leaf=10",
        DecisionTreeRegressor(
            max_depth=10,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Decision Tree depth=18 leaf=10",
        DecisionTreeRegressor(
            max_depth=18,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Decision Tree depth=24 leaf=10",
        DecisionTreeRegressor(
            max_depth=24,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Decision Tree depth=None leaf=10",
        DecisionTreeRegressor(
            max_depth=None,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Random Forest 100 trees depth=15 leaf=5",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Random Forest 100 trees depth=25 leaf=5",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=25,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Random Forest 100 trees depth=None leaf=5",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=None,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    )
]

In [ ]:
# ============================================================
# 6. Validation comparison
# ============================================================

validation_results = []
fitted_candidates = {}

for model_name, model in candidate_models:

    print("=" * 70)
    print("Training:", model_name)

    result, fitted_model = fit_and_evaluate(
        model=model,
        model_name=model_name,
        X_fit=X_development,
        y_fit=y_development,
        X_eval=X_validation,
        y_eval=y_validation
    )

    validation_results.append(result)
    fitted_candidates[model_name] = fitted_model

    print("Train R²:", result["Train R2"])
    print("Validation R²:", result["Validation R2"])
    print("Validation RMSE:", result["Validation RMSE"])
    print("Validation MdAPE:", result["Validation MdAPE"])
    print("Fit seconds:", result["Fit Seconds"])


validation_results = pd.DataFrame(
    validation_results
).sort_values(
    "Validation R2",
    ascending=False
).reset_index(drop=True)

print("\nValidation model comparison:")
display(validation_results)

Training: Baseline Mean
Train R²: 0.0
Validation R²: -1.4849937130856361e-05
Validation RMSE: 1402909.886857194
Validation MdAPE: 55.38057019985534
Fit seconds: 0.00506901741027832
Training: Week 4 Linear Regression
Train R²: 0.48651586968798366
Validation R²: 0.45802633546136806
Validation RMSE: 1032798.6063706204
Validation MdAPE: 35.087147953490394
Fit seconds: 10.881087064743042
Training: Decision Tree depth=10 leaf=10
Train R²: 0.7622672582232943
Validation R²: 0.6079024163573004
Validation RMSE: 878463.3237842846
Validation MdAPE: 17.439423043351752
Fit seconds: 3.1550099849700928
Training: Decision Tree depth=18 leaf=10
Train R²: 0.830947233814519
Validation R²: 0.6454042090488649
Validation RMSE: 835397.8233970786
Validation MdAPE: 11.794992713137397
Fit seconds: 4.145069122314453
Training: Decision Tree depth=24 leaf=10
Train R²: 0.8327248620978768
Validation R²: 0.646035111187306
Validation RMSE: 834654.3163798115
Validation MdAPE: 11.651417142857145
Fit seconds: 3.7296230792

,Model,Train R2,Validation R2,Validation RMSE,Validation MAE,Validation MAPE,Validation MdAPE,Fit Seconds
0,Decision Tree depth=None leaf=10,0.8328,0.6461,"834,559.1000","277,477.6673",19.1188,11.6596,3.7368
1,Decision Tree depth=24 leaf=10,0.8327,0.6460,"834,654.3164","277,670.0832",19.1537,11.6514,3.7296
2,Decision Tree depth=18 leaf=10,0.8309,0.6454,"835,397.8234","279,283.8717",19.3872,11.7950,4.1451
3,Decision Tree depth=10 leaf=10,0.7623,0.6079,"878,463.3238","339,057.5992",25.8787,17.4394,3.1550
4,Random Forest 100 trees depth=None leaf=5,0.6057,0.5279,"963,974.0151","401,829.4630",39.3224,27.3880,13.6018
5,Random Forest 100 trees depth=25 leaf=5,0.5745,0.4973,"994,722.9621","429,511.7010",43.2911,30.3109,11.1926
6,Week 4 Linear Regression,0.4865,0.4580,"1,032,798.6064","534,515.3559",50.7893,35.0871,10.8811
7,Random Forest 100 trees depth=15 leaf=5,0.5012,0.4386,"1,051,159.9084","490,133.5968",51.9761,36.1603,9.0244
8,Baseline Mean,0.0000,-0.0000,"1,402,909.8869","724,864.4429",80.5238,55.3806,0.0051


In [41]:
validation_display = validation_results.copy()

for col in [
    "Validation RMSE",
    "Validation MAE"
]:
    validation_display[col] = validation_display[col].map(
        lambda x: f"${x:,.0f}"
    )

for col in [
    "Validation MAPE",
    "Validation MdAPE"
]:
    validation_display[col] = validation_display[col].map(
        lambda x: f"{x:.2f}%"
    )

for col in [
    "Train R2",
    "Validation R2"
]:
    validation_display[col] = validation_display[col].map(
        lambda x: f"{x:.4f}"
    )

display(validation_display)

,Model,Train R2,Validation R2,Validation RMSE,Validation MAE,Validation MAPE,Validation MdAPE,Fit Seconds
0,Decision Tree depth=None leaf=10,0.8328,0.6461,"$834,559","$277,478",19.12%,11.66%,3.7368
1,Decision Tree depth=24 leaf=10,0.8327,0.6460,"$834,654","$277,670",19.15%,11.65%,3.7296
2,Decision Tree depth=18 leaf=10,0.8309,0.6454,"$835,398","$279,284",19.39%,11.79%,4.1451
3,Decision Tree depth=10 leaf=10,0.7623,0.6079,"$878,463","$339,058",25.88%,17.44%,3.1550
4,Random Forest 100 trees depth=None leaf=5,0.6057,0.5279,"$963,974","$401,829",39.32%,27.39%,13.6018
5,Random Forest 100 trees depth=25 leaf=5,0.5745,0.4973,"$994,723","$429,512",43.29%,30.31%,11.1926
6,Week 4 Linear Regression,0.4865,0.4580,"$1,032,799","$534,515",50.79%,35.09%,10.8811
7,Random Forest 100 trees depth=15 leaf=5,0.5012,0.4386,"$1,051,160","$490,134",51.98%,36.16%,9.0244
8,Baseline Mean,0.0000,-0.0000,"$1,402,910","$724,864",80.52%,55.38%,0.0051


In [ ]:
# ============================================================
# 7. Select the best tree and forest using validation R²
# ============================================================

best_tree_name = (
    validation_results[
        validation_results["Model"].str.startswith(
            "Decision Tree"
        )
    ]
    .iloc[0]["Model"]
)

best_forest_name = (
    validation_results[
        validation_results["Model"].str.startswith(
            "Random Forest"
        )
    ]
    .iloc[0]["Model"]
)

print("Best Decision Tree:", best_tree_name)
print("Best Random Forest:", best_forest_name)

Best Decision Tree: Decision Tree depth=None leaf=10
Best Random Forest: Random Forest 100 trees depth=None leaf=5


In [43]:
best_tree_params = fitted_candidates[
    best_tree_name
].get_params()

best_forest_params = fitted_candidates[
    best_forest_name
].get_params()

final_models = [
    (
        "Baseline Mean",
        DummyRegressor(
            strategy="mean"
        )
    ),
    
    (
        "Week 4 Linear Regression",
        LinearRegression()
    ),

    (
        best_tree_name,
        DecisionTreeRegressor(
            **best_tree_params
        )
    ),

    (
        best_forest_name,
        RandomForestRegressor(
            **best_forest_params
        )
    )
]

In [ ]:
# ============================================================
# 8. Final test evaluation
# ============================================================

test_results = []
final_fitted_models = {}
test_predictions = {}

for model_name, model in final_models:

    print("=" * 70)
    print("Final training:", model_name)

    start_time = time.time()

    model.fit(
        X_train,
        y_train
    )

    fit_seconds = time.time() - start_time

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_metrics = calculate_metrics(
        y_train,
        train_pred
    )

    test_metrics = calculate_metrics(
        y_test,
        test_pred
    )

    test_results.append({
        "Model": model_name,
        "Train R2": train_metrics["R2"],
        "Test R2": test_metrics["R2"],
        "Test RMSE": test_metrics["RMSE"],
        "Test MAE": test_metrics["MAE"],
        "Test MAPE": test_metrics["MAPE"],
        "Test MdAPE": test_metrics["MdAPE"],
        "Fit Seconds": fit_seconds
    })

    final_fitted_models[model_name] = model
    test_predictions[model_name] = test_pred

    print("Train R²:", train_metrics["R2"])
    print("Test R²:", test_metrics["R2"])
    print("Test RMSE:", test_metrics["RMSE"])
    print("Test MAE:", test_metrics["MAE"])
    print("Test MAPE:", test_metrics["MAPE"])
    print("Test MdAPE:", test_metrics["MdAPE"])


test_results = pd.DataFrame(
    test_results
).sort_values(
    "Test R2",
    ascending=False
).reset_index(drop=True)

print("\nFinal test comparison:")
display(test_results)

Final training: Baseline Mean
Train R²: 0.0
Test R²: -0.0015186344638324911
Test RMSE: 1679718.9917059606
Test MAE: 737644.9731731376
Test MAPE: 80.52583367549015
Test MdAPE: 51.684433075395894
Final training: Week 4 Linear Regression
Train R²: 0.4804408364181886
Test R²: 0.3056315873685106
Test RMSE: 1398627.5979322179
Test MAE: 555080.9802494604
Test MAPE: 55.43940130496283
Test MdAPE: 34.53159780349499
Final training: Decision Tree depth=None leaf=10
Train R²: 0.8359165879639432
Test R²: 0.4735875032503336
Test RMSE: 1217783.8515002907
Test MAE: 285487.9527645561
Test MAPE: 20.339345325265686
Test MdAPE: 11.636506313089864
Final training: Random Forest 100 trees depth=None leaf=5
Train R²: 0.6019787304274127
Test R²: 0.3611357396975301
Test RMSE: 1341564.0504619386
Test MAE: 437620.3284109412
Test MAPE: 46.81555546801853
Test MdAPE: 29.483558905272886

Final test comparison:


,Model,Train R2,Test R2,Test RMSE,Test MAE,Test MAPE,Test MdAPE,Fit Seconds
0,Decision Tree depth=None leaf=10,0.8359,0.4736,"1,217,783.8515","285,487.9528",20.3393,11.6365,5.9020
1,Random Forest 100 trees depth=None leaf=5,0.6020,0.3611,"1,341,564.0505","437,620.3284",46.8156,29.4836,19.7455
2,Week 4 Linear Regression,0.4804,0.3056,"1,398,627.5979","555,080.9802",55.4394,34.5316,9.9858
3,Baseline Mean,0.0000,-0.0015,"1,679,718.9917","737,644.9732",80.5258,51.6844,0.0031


In [ ]:
# ============================================================
# 9. Final test evaluation
# ============================================================

test_results = []
final_fitted_models = {}
test_predictions = {}

for model_name, model in final_models:

    print("=" * 70)
    print("Final training:", model_name)

    start_time = time.time()

    model.fit(
        X_train,
        y_train
    )

    fit_seconds = time.time() - start_time

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_metrics = calculate_metrics(
        y_train,
        train_pred
    )

    test_metrics = calculate_metrics(
        y_test,
        test_pred
    )

    test_results.append({
        "Model": model_name,
        "Train R2": train_metrics["R2"],
        "Test R2": test_metrics["R2"],
        "Test RMSE": test_metrics["RMSE"],
        "Test MAE": test_metrics["MAE"],
        "Test MAPE": test_metrics["MAPE"],
        "Test MdAPE": test_metrics["MdAPE"],
        "Fit Seconds": fit_seconds
    })

    final_fitted_models[model_name] = model
    test_predictions[model_name] = test_pred

    print("Train R²:", train_metrics["R2"])
    print("Test R²:", test_metrics["R2"])
    print("Test RMSE:", test_metrics["RMSE"])
    print("Test MAE:", test_metrics["MAE"])
    print("Test MAPE:", test_metrics["MAPE"])
    print("Test MdAPE:", test_metrics["MdAPE"])
    print("Fit seconds:", fit_seconds)


test_results = pd.DataFrame(
    test_results
).sort_values(
    "Test R2",
    ascending=False
).reset_index(drop=True)

display(test_results)

Final training: Baseline Mean
Train R²: 0.0
Test R²: -0.0015186344638324911
Test RMSE: 1679718.9917059606
Test MAE: 737644.9731731376
Test MAPE: 80.52583367549015
Test MdAPE: 51.684433075395894
Fit seconds: 0.007361888885498047
Final training: Week 4 Linear Regression


Train R²: 0.4804408364181886
Test R²: 0.3056315873685106
Test RMSE: 1398627.5979322179
Test MAE: 555080.9802494604
Test MAPE: 55.43940130496283
Test MdAPE: 34.53159780349499
Fit seconds: 12.20389986038208
Final training: Decision Tree depth=None leaf=10
Train R²: 0.8359165879639432
Test R²: 0.4735875032503336
Test RMSE: 1217783.8515002907
Test MAE: 285487.9527645561
Test MAPE: 20.339345325265686
Test MdAPE: 11.636506313089864
Fit seconds: 5.007088899612427
Final training: Random Forest 100 trees depth=None leaf=5
Train R²: 0.6019787304274127
Test R²: 0.3611357396975301
Test RMSE: 1341564.0504619386
Test MAE: 437620.3284109412
Test MAPE: 46.81555546801853
Test MdAPE: 29.483558905272886
Fit seconds: 23.081439971923828


,Model,Train R2,Test R2,Test RMSE,Test MAE,Test MAPE,Test MdAPE,Fit Seconds
0,Decision Tree depth=None leaf=10,0.8359,0.4736,"1,217,783.8515","285,487.9528",20.3393,11.6365,5.0071
1,Random Forest 100 trees depth=None leaf=5,0.6020,0.3611,"1,341,564.0505","437,620.3284",46.8156,29.4836,23.0814
2,Week 4 Linear Regression,0.4804,0.3056,"1,398,627.5979","555,080.9802",55.4394,34.5316,12.2039
3,Baseline Mean,0.0000,-0.0015,"1,679,718.9917","737,644.9732",80.5258,51.6844,0.0074


In [ ]:
# ============================================================
# 10. Prediction comparison
# ============================================================

best_model_name = test_results.iloc[0]["Model"]
best_test_pred = test_predictions[best_model_name]

prediction_comparison = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": best_test_pred
})

prediction_comparison["Error"] = (
    prediction_comparison["Predicted"] -
    prediction_comparison["Actual"]
)

prediction_comparison["AbsoluteError"] = (
    prediction_comparison["Error"].abs()
)

prediction_comparison["AbsolutePercentageError"] = (
    prediction_comparison["AbsoluteError"] /
    prediction_comparison["Actual"].abs()
) * 100

print("Best model:", best_model_name)

display(
    prediction_comparison.sample(
        20,
        random_state=RANDOM_STATE
    )
)

Best model: Decision Tree depth=None leaf=10


,Actual,Predicted,Error,AbsoluteError,AbsolutePercentageError
456,"372,000.0000","305,081.2500","-66,918.7500","66,918.7500",17.9889
3363,"739,000.0000","716,416.6667","-22,583.3333","22,583.3333",3.0559
11928,"720,000.0000","801,538.1176","81,538.1176","81,538.1176",11.3247
9624,"1,425,000.0000","2,102,038.4615","677,038.4615","677,038.4615",47.5115
4925,"460,000.0000","433,600.0000","-26,400.0000","26,400.0000",5.7391
525,"640,000.0000","757,718.1818","117,718.1818","117,718.1818",18.3935
7180,"865,000.0000","918,636.3636","53,636.3636","53,636.3636",6.2007
6268,"1,510,000.0000","1,582,843.7500","72,843.7500","72,843.7500",4.8241
3731,"315,000.0000","320,010.0000","5,010.0000","5,010.0000",1.5905
7943,"3,334,475.0000","2,936,720.5000","-397,754.5000","397,754.5000",11.9285


In [48]:
print("Largest absolute errors:")

display(
    prediction_comparison.sort_values(
        "AbsoluteError",
        ascending=False
    ).head(20)
)

Largest absolute errors:


,Actual,Predicted,Error,AbsoluteError,AbsolutePercentageError
8629,"97,972,500.0000","974,811.0000","-96,997,689.0000","96,997,689.0000",99.0050
5513,"48,720,000.0000","2,166,135.3158","-46,553,864.6842","46,553,864.6842",95.5539
9241,"35,000,000.0000","4,212,920.0000","-30,787,080.0000","30,787,080.0000",87.9631
1884,"28,000,000.0000","4,768,057.2727","-23,231,942.7273","23,231,942.7273",82.9712
2097,"22,500,000.0000","9,022,222.1667","-13,477,777.8333","13,477,777.8333",59.9012
5290,"16,250,000.0000","4,803,500.0000","-11,446,500.0000","11,446,500.0000",70.4400
11803,"15,900,000.0000","5,907,090.9091","-9,992,909.0909","9,992,909.0909",62.8485
10182,"11,150,000.0000","1,237,416.6667","-9,912,583.3333","9,912,583.3333",88.9021
2765,"18,277,380.0000","9,022,222.1667","-9,255,157.8333","9,255,157.8333",50.6372
6759,"21,450,000.0000","12,211,666.6667","-9,238,333.3333","9,238,333.3333",43.0692


The model still performs poorly on luxury properties and often severely underpredicts homes above $10 million. These rare high-end sales create very large errors and reduce R². Future improvements could use a separate luxury-home model, log-transformed prices, and more luxury-specific features.

## Interpretation

1. Best validation R²: Decision Tree (depth=None, leaf=10) at 0.6461.

2. Best typical percentage error: Decision Tree (depth=24, leaf=10) at 11.65% MdAPE, although the unrestricted tree was nearly identical at 11.66%.

3. The Week 4 Linear Regression achieved a validation R² of 0.4580 and a MdAPE of 35.09%. This provides a stronger baseline than the mean-only model, but its performance suggests that the relationship between property characteristics and ClosePrice is not fully linear.

4. The Decision Tree models substantially outperform both the Linear Regression and mean baseline. The best tree improves validation R² from 0.4580 to 0.6461 and reduces MdAPE from 35.09% to about 11.66%, indicating that nonlinear splits and feature interactions are highly useful for housing-price prediction.

5. The depth-24 and unrestricted Decision Trees perform almost identically. This suggests that increasing depth beyond approximately 24 provides very little additional validation benefit. The unrestricted tree also shows a noticeable train-validation gap, with training R² of 0.8328 versus validation R² of 0.6461, indicating some overfitting.

6. The Random Forest models perform better than Linear Regression in some configurations, but they underperform the single Decision Tree in this experiment. The best Random Forest reaches a validation R² of 0.5279 with a MdAPE of 27.39%. Its relatively low training R² suggests that the current settings—particularly max_features="sqrt" and min_samples_leaf=5—may be too restrictive and may cause underfitting.

Overall, the best Week 5 validation result comes from the Decision Tree with no maximum depth and a minimum leaf size of 10. However, the depth-24 tree may be preferable because it achieves almost the same performance while providing slightly stronger regularization and a marginally better MdAPE.

Next iterations should test less restrictive Random Forest settings, such as a larger max_features value and smaller leaf sizes, while continuing to compare all models against the same Week 4 Linear Regression baseline.